# CB pricing

전환사채는 투자자에게 해당 사채를 (해당 사채의 액면가) / (전환가액) 만큼의 주식으로 전환할 수 있는 전환권이 주어집니다.<br>
또, 전환사채를 일부 조기상환할 수 있는 콜옵션이 발행자에게, 원금을 보전할 수 있는 풋옵션이 투자자에게 부여됩니다.<br>
따라서 LSMC를 이용해서 전환사채를 평가할 수 있습니다.<br>
전환사채 평가에서 중요한 점은 다음과 같습니다.
1. 콜옵션이 끝나는 날까지 콜옵션의 행사가능비율만큼은 전환이 불가능합니다. 따라서 대부분의 경우 두 번에 걸친 상환이 이루어집니다.
2. 1에서 나아가 콜옵션은 전환권, 풋옵션보다 우선됩니다. 따라서 콜옵션의 행사 여부는 투자자가 모든 권리를 행사한 후 마지막에 결정합니다.
3. 콜옵션과 풋옵션의 행사일은 대체로 전환가능구간에 걸쳐 있습니다. 따라서 경제주체별로 여러 가지 권리를 비교하여 그중 이익이 제일 행사가치를 택해야 합니다.
4. 대체로 마지막 콜옵션의 행사일과 첫 풋옵션의 행사일이 동일합니다. 따라서 해당 날짜에서는 전환권과 콜옵션, 풋옵션이 모두 행사 가능하기에 각 옵션의 행사가치를 비교하는 로직이 필요합니다.
5. 전환권은 보통 전환가능구간 중 어느 날이든 행사가 가능합니다. 하지만 매일 LSMC를 실행할 경우 시간이 너무 오래 걸리기에, LSMC는 전환권을 제외한 옵션이 존재하는 날, 전환가액이 바뀌는 날에 실행하고, 두 케이스에 속하지 않은 경우 7일에 한 번씩 실행합니다.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

우선 편의를 위해 다음과 같은 가정을 세웁니다<br>
1. 무위험이자율의 장기평균은 3%이고, 할인이자율의 장기평균은 6%이다.
2. 두 이자율의 Hull-White vol은 0.5%로, 주식의 변동성은 20%로 고정한다.
그리고 다음과 같이 설정합니다.<br>
$$S_0 = 10000$$
$$NA = 100000$$
$$dt = {{1}\over{365}}$$
$$T = 10$$
$$r_{coupon} = 0$$
$$r_{riskfree} = 0.03$$
$$r_{discount} = 0.06$$
전환청구기간은 $T=1$부터 $T = 9.5$까지이며, 콜옵션은 $T = 2$부터 $T = 5$까지 1년 간격으로 총 4번 존재하며,<br>
행사가는 액면가에 단리이자 3%를 적용한 원리합계입니다. 풋옵션은 $T = 5$부터 $T = 9$까지 1년 간격으로 총 5번 존재하며, 행사가는 액면가입니다.<br>
만기 지급시, 이자를 30% 지급하여 130000원으로 상환합니다. 전환가액은 10000원으로 시작해서, 3년마다 변경이 이루어지는데, 다음 프로세스를 따릅니다.<br>
30일 주가평균, 7일 주가평균, 전일 주가 의 평균과 전일 주가 중 낮은 값을 새로운 전환가액으로 삼되, 전환가액은 7000원 이상 10000원 이하입니다.<br>
콜옵션은 전환사채의 30%까지 행사가 가능하며, 마지막 콜옵션의 행사일인 $T = 5$까지 채권자는 해당 비율의 전환사채를 비전환 상태로 유지해야 합니다.<br>


In [2]:
def refixing(path, T, original):
    monthmean = np.mean(path.loc[:,T-30:T],axis=1)
    weekmean = np.mean(path.loc[:,T-7:T],axis=1)
    yesterday = path.loc[:,T-1]
    mean = (monthmean + weekmean + yesterday)/3
    refix = mean + (mean>yesterday) * (mean-yesterday)
    refix += (refix < 0.7*original) * (0.7*original - refix)
    refix += (refix > original) * (original - refix)
    return refix

def next_shortrate(rt, kappa, sigma, theta, dt, dW):
    drift = kappa*(theta - rt)*dt
    rand = sigma * np.sqrt(dt) * dW
    return rt + drift + rand

def next_stockprice(St, rt, sigma, dt, dW):
    drift = rt*dt
    rand = sigma*np.sqrt(dt)*dW
    return St + St*(drift + rand)

def CalcDF(shortrate, start, end, dt):
    DF = 1.0
    for i in range(start,end):
        r = shortrate[i]
        shortDF = np.exp(-r*dt)
        DF *= shortDF
    return DF

dt = 1/365
T = 10
rf = 0.03
riskfree_theta = 0.03
rd = 0.06
discount_theta = 0.06
NTrial = 10000
St = 10000
NA = 100000
kappa = 0.01
HWVol = 0.005
StockVol = 0.2


In [3]:
Stock = pd.DataFrame(index = range(1,NTrial+1), columns = range(0,int(T/dt)+1))
Riskfreerate = pd.DataFrame(index = range(1,NTrial+1), columns = range(0,int(T/dt)+1))
Discountrate = pd.DataFrame(index = range(1,NTrial+1), columns = range(0,int(T/dt)+1))
RefixingValue = pd.DataFrame(index = range(1,NTrial+1),columns = range(0,10,3))
Stock[0] = St
Riskfreerate[0] = rf
Discountrate[0] = rd
RefixingValue[0] = 10000
StockRand = np.random.normal(size=(int(T/dt), NTrial))
RfRand = np.random.normal(size=(int(T/dt), NTrial))
RdRand = np.random.normal(size=(int(T/dt), NTrial))

In [4]:
for i in range(0,int(T/dt)):
    Riskfreerate[i+1] = next_shortrate(Riskfreerate[i], kappa, HWVol, riskfree_theta, dt, RfRand[i])
    Discountrate[i+1] = next_shortrate(Discountrate[i], kappa, HWVol, discount_theta, dt, RdRand[i])
    Stock[i+1] = next_stockprice(Stock[i],Riskfreerate[i],StockVol,dt,StockRand[i])
NumRefixing = int(T/3)
for i in range(0,NumRefixing):
    RefixingValue[3*(i+1)] = refixing(Stock, 365*3*(i+1), 10000)

주가 path와 각 path별 전환가액을 모두 산출했으니 이제 LSMC에서 행사될 옵션을 선택하는 로직을 제작해야 합니다.<Br>
우선 전환 가능 구간의 모든 날짜는 다음 유형으로 분리가 가능합니다.
1. 전환권과 풋옵션이 행사 가능한 날
2. 전환권과 콜옵션이 행사 가능한 날
3. 전환권과 풋옵션과 콜옵션이 행사 가능한 날(마지막 콜옵션 행사일 = 첫 풋옵션 행사일)
4. 전환권이 행사 가능한 날

4의 경우는 단순하게 보유가치보다 전환가치가 크다면 해당 시점에서 전환권을 행사하면 됩니다.<br>
1의 경우 전환권과 풋옵션이 모두 투자자의 권리이이게 전환가치와 풋옵션 행사가치를 비교하여 큰 값이 보유가치보다 크다면 해당 옵션을 행사합니다.<br>
2의 경우 전환권은 투자자의 권리이지만 콜옵션은 발행자의 권리입니다. 그리고 콜옵션이 전환권보다 우선되기에, 다음과 같은 알고리즘을 따릅니다.<br>
2-1. 보유가치보다 전환가치가 클 경우 전환 결정, 작다면 보유 결정
2-2. 1에서 정해진 가치보다 1에서 행사가 결정된 옵션과 콜옵션을 동시에 행사했을 때의 가치가 더 작다면 콜옵션 행사 결정
3의 경우에는 다음과 같습니다.<br>
3-1. 보유가치와 전환가치, 풋옵션 행사가치 중 가장 큰 가치를 계산하여 해당 권리 행사
3-2. 1에서 정해진 가치보다 1에서 행사가 결정된 옵션과 콜옵션을 동시에 행사했을 때의 가치가 더 작다면 콜옵션 행사 결정

위에서 설명했듯이 콜옵션으로 인해 의무보유구간이 존재하기에 두 번의 상환 스케줄이 생깁니다.<br>
행사가치를 구하기 위해서는 다음과 같은 알고리즘을 거칩니다.<br>

### 전환가치 계산
1. 해당 시점이 마지막 콜옵션 행사일보다 나중인 경우, 보유의무가 소멸했기에 채권 전체를 전환 가능.<br>

   전환가치 = 해당 시점 주가 $\times$ 전환 주식 수 (액면가 / 전환가액)<br>
   
2. 해당 시점이 마지막 콜옵션 행사일보다 이전일 경우, 보유의무가 없는 부분만 전환 가능.<br>

   전환가치 = (콜옵션 행사가능 비율) $\times$ (보유가치) + (1 - 콜옵션 행사가능 비율) $\times$ 전환 주식 수<br>


### 콜옵션 행사가치 계산
콜옵션의 경우 제일 우선되기에 모두 아래 식으로 계산할 수 있습니다.<br>

   행사가치 = (콜옵션 행사가능 비율) $\times$ (콜옵션 행사가) +  (1 - 콜옵션 행사가능 비율) $\times$ (보유가치)

이때 보유가치는 해당 날짜에 투자자의 권리가 행사된 경우에는 해당 가치로 교체됩니다.

### 풋옵션 행사가치 계산
풋옵션의 경우 첫 풋옵션의 행사일이 마지막 콜옵션의 행사일과 일치하기에, 모든 행사 시점에서 보유의무가 존재하지 않습니다.<br>
따라서 풋옵션의 행사가치는 풋옵션의 행사가입니다.

위 사항을 종합하여 최적 권리 행사 함수를 제작합니다.

In [5]:
def CallValue(CallStrike, #콜옵션 행사가
              CallRatio, #콜옵션 행사가능비율
              NextRedemptionValue): #이후행사가치
    return CallStrike * CallRatio + NextRedemptionValue * (1 - CallRatio)

def ConvertValue( #전환가치 계산
    StockPrice, #현재주가
    RefixingValue, #전환가액
    NotionalAmount): #액면가
    return StockPrice * (NotionalAmount / RefixingValue)


def CalcOptimal_CallNConvert( #전환권 + 콜옵션일 때 최적 행사 권리 계산
    RegressionValue, #보유가치(회귀분석값) 
    CallStrike, #콜옵션 행사가
    CallRatio, #콜옵션 행사가능비율
    StockPrice, #현재주가
    RefixingValue, #전환가액
    NotionalAmount
    
):
    if(RegressionValue < ConvertValueALC(StockPrice,RefixingValue,NotionalAmount)): #전환가치 > 보유가치
        if(ConvertValue(StockPrice,RefixingValue,NotionalAmount) > 
           CallValue(CallStrike,CallRatio, ConvertValue(StockPrice,RefixingValue,NotionalAmount))): #전환 및 행사가치 < 전환치치
            return "CallNConvert"
        else:
            return "Convert"
    else:
        if(CallStrike<RegressionValue): #행사가치 < 보유가치
            return "Call"
        else:
            return "Non"
            
def CalcOptimal_PutNConvert( #전환권 + 풋옵션일 때 최적 행사 권리 계산
    RegressionValue, #보유가치(회귀분석값) 
    PutStrike, #풋옵션 행사가
    StockPrice, #현재주가
    RefixingValue, #전환가액
    NotionalAmount): #액면가
    #풋 vs 전환
    if(ConvertValue(StockPrice,RefixingValue,NotionalAmount) < PutStrike):
        if(RegressionValue < PutStrike):
            return "Put"
        else:
            return "Non"
    else:
        if(RegressionValue < ConvertValue(StockPrice,RefixingValue,NotionalAmount)):
            return "Convert"
        else:
            return "Non"

def CalcOptimal_CallNPutNConvert( #전환권 + 풋옵션 + 콜옵션일 때 최적 행사 권리 계산
    RegressionValue, #보유가치(회귀분석값) 
    CallStrike, #콜옵션 행사가
    CallRatio, #콜옵션 행사가능비율
    PutStrike, #풋옵션 행사가
    StockPrice, #현재주가
    RefixingValue, #전환가액
    NotionalAmount): #액면가
    #풋 vs 전환
    if(ConvertValue(StockPrice,RefixingValue,NotionalAmount) < PutStrike):
        if(RegressionValue < PutStrike):
            if(PutStrike > CallValue(CallStrike,CallRatio,PutStrike)):
                return "CallNPut"
            else:
                return "Put"
        else:
            if(RegressionValue > CallStrike):
                return "Call"
            else:
                return "Non"
    else:
        if(RegressionValue < ConvertValue(StockPrice,RefixingValue,NotionalAmount)):
            if(ConvertValue(StockPrice,RefixingValue,NotionalAmount) > 
               CallValue(CallStrike,CallRatio,ConvertValue(StockPrice,RefixingValue,NotionalAmount))):
                return "CallNConvert"
            else:
                return "Convert"
        else:
            if(RegressionValue > CallStrike):
                return "Call"
            else:
                return "Non"

def CalcOptimal_Convert(
    RegressionValue, #보유가치(회귀분석값) 
    StockPrice, #현재주가
    RefixingValue, #전환가액
    NotionalAmount): #액면가
    if(RegressionValue < ConvertValue(StockPrice,RefixingValue,NotionalAmount)):
        return "Convert"
    else:
        return "Non"

def CalcContiValue(T,CallRatio, RiskFree, Discount, DutyRedemption, NDutyRedemption):
    DutyContiValue = DutyRedemption["Value"] * CalcDF(Discount,T,DutyRedemption["T"],dt) * (1 * DutyRedemption["Div"] != 1)
    DutyContiValue += DutyRedemption["Value"] * CalcDF(RiskFree,T,DutyRedemption["T"],dt) * (1 * DutyRedemption["Div"] == 1)
    NDutyContiValue = NDutyRedemption["Value"] * CalcDF(Discount,T,NDutyRedemption["T"],dt) * (1 * NDutyRedemption["Div"] != 1)
    NDutyContiValue += NDutyRedemption["Value"] * CalcDF(RiskFree,T,NDutyRedemption["T"],dt) * (1 * NDutyRedemption["Div"] == 1)
    return DutyContiValue * CallRatio + NDutyContiValue *(1-CallRatio)

이제 각 path별 상환스케줄과 해당 시점을 저장하는 배열을 두개 만듭니다.<br>
첫 번째 배열은 보유의무채권의 상환스케줄이고, 두 번째 배열은 의무가 없는 채권의 상환스케줄입니다.<br>
상환스케줄에서 숫자는 다음과 같은 의미를 가집니다.<br>
0: 만기상환<br>
1: 전환<br>
2: 콜옵션 행사<br>
3: 풋옵션 행사<br>

In [6]:
DutyRedemption = pd.DataFrame(index = range(1,NTrial+1), columns = ["Div", "T", "Value"])
NDutyRedemption = pd.DataFrame(index = range(1,NTrial+1), columns = ["Div", "T", "Value"])

In [7]:
DutyRedemption["Div"] = 0
DutyRedemption["T"] = 3650
DutyRedemption["Value"] = NA
NDutyRedemption["Div"] = 0
NDutyRedemption["T"] = 3650
NDutyRedemption["Value"] = NA
CallTArray = [365*2, 365*3, 365*4, 365*5]
CallStrikeArray = [NA * (1+2*0.03), NA * (1+3*0.03), NA * (1+4*0.03), NA * (1+5*0.03)]
PutTArray = [365*5, 365*6, 365*7, 365*8, 365*9]
RefixingT = [0, 365*3, 365*6, 365*9]
PutStrike = NA
NthCall = 3
NthPut = 4
NthRefixing = 3

이제 행사여부 판단 함수와 LSMC를 이용해서 상환스케줄을 변경해 갑니다. LSMC에서 독립변수는 해당 시점 주가와 단기금리입니다.

In [8]:
ConvertEnd = int(9.5*365)
ConvertStart = 365
for i in range(ConvertEnd,ConvertStart,-1):
    if(i==CallTArray[NthCall] and i==PutTArray[NthPut]):
        CallStrike = CallStrikeArray[NthCall]
        ConvertPrice = RefixingValue.iloc[:,NthRefixing]
        NthCall-=1
        NthPut-=1
    elif(i==PutTArray[NthPut]):
        ConvertPrice = RefixingValue.iloc[:,NthRefixing]
        NthPut-=1
    elif(i==CallTArray[NthCall]):
        CallStrike = CallStrikeArray[NthCall]
        ConvertPrice = RefixingValue.iloc[:,NthRefixing]
        NthCall-=1
    elif((i-ConvertStart)%7==0):
        ConvertPrice = RefixingValue.iloc[:,NthRefixing]
    
    if(i==RefixingT[NthRefixing]):
        NthRefixing-=1